# schedule_analysis_manifest_preflight_v1

目标：验证 `src.schedule` 在 dashboard refresh 中可以先检查 analysis manifest 状态；默认只提示，显式 auto backfill 时才生成 `analysis_manifest.json`。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = REPO_ROOT / 'research/schedule_analysis_manifest_preflight_v1'
AUDIT_CARD = CARD / 'audit_card'
AUTO_CARD = CARD / 'auto_card'
LOCKS = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)
{'card': str(CARD.relative_to(REPO_ROOT))}

{'card': 'research/schedule_analysis_manifest_preflight_v1'}

## 模拟旧 matrix card

matrix runner 只写旧固定 sidecar，并额外写入 `predictions.csv`，用于验证 schedule preflight 不把私有明细放进 dashboard manifest。

In [2]:
def legacy_matrix_runner(*, card_dir: Path, start: str, end: str, train_end: str, model_names, top_n_factors: int):
    card_dir.mkdir(parents=True, exist_ok=True)
    (card_dir / 'analysis_summary.json').write_text(json.dumps({
        'schema_version': 'analysis.report.v1',
        'status': 'legacy_matrix_completed',
        'start': start,
        'end': end,
        'train_end': train_end,
        'model_count': len(tuple(model_names)),
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    pd.DataFrame([
        {'rank': 1, 'model': 'ridge_mse', 'return_pct': 2.8, 'relative_return_pct': 1.1, 'stability_score': 2.2},
        {'rank': 2, 'model': 'gcn_mse', 'return_pct': 1.0, 'relative_return_pct': -0.5, 'stability_score': 0.4},
    ]).to_csv(card_dir / 'model_comparison.csv', index=False)
    pd.DataFrame([
        {'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 3.4},
        {'model': 'ridge_mse', 'regime': 'stress', 'return_pct': -0.8},
    ]).to_csv(card_dir / 'regime_metrics.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'window': 'test_2026', 'return_pct': 2.8}]).to_csv(card_dir / 'window_metrics.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'return_kind': 'strategy_return', 'annualized_return': 0.2}]).to_csv(card_dir / 'qlib_risk_metrics.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'segment_type': 'all', 'segment': 'all', 'diagnosis': '基准模型。'}]).to_csv(card_dir / 'model_effect_deltas.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'prediction': 0.1}]).to_csv(card_dir / 'predictions.csv', index=False)
    return {'status': 'feature_set_matrix_completed', 'variant_count': len(tuple(model_names))}


## 审计模式：只提示不写 manifest

开启 `analysis_manifest_preflight=True` 但不打开 auto backfill，schedule payload 应报告 `needs_manifest`，旧 card 不被改写。

In [3]:
audit_config = ModelDashboardPipelineConfig(
    name='schedule_manifest_audit',
    card_dir=AUDIT_CARD,
    dashboard_output_path=CARD / 'audit_dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    analysis_manifest_preflight=True,
)
audit_result = run_model_dashboard_refresh_once(
    audit_config,
    lock_dir=LOCKS,
    matrix_runner=legacy_matrix_runner,
)
assert audit_result.status == 'success'
assert audit_result.payload['analysis_manifest_preflight_status'] == 'needs_manifest'
assert not (AUDIT_CARD / 'analysis_manifest.json').exists()
audit_result.payload

{'matrix_card': '/Users/rziiiii/dev/workspace/research/schedule_analysis_manifest_preflight_v1/audit_card',
 'dashboard_export': '/Users/rziiiii/dev/workspace/research/schedule_analysis_manifest_preflight_v1/audit_dashboard_export.json',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_artifact_id': 'audit_card',
 'dashboard_manifest_kind': 'fixed_sidecars',
 'dashboard_manifest_path': '/Users/rziiiii/dev/workspace/research/schedule_analysis_manifest_preflight_v1/audit_card',
 'dashboard_manifest_hash': '',
 'dashboard_manifest_producer': '',
 'dashboard_page': '',
 'dashboard_page_schema_version': '',
 'dashboard_view_model_schema_version': '',
 'dashboard_page_model_card_count': 0,
 'top_model': 'ridge_mse',
 'attribution_card': '',
 'attribution_status': 'not_provided',
 'matrix_status': 'feature_set_matrix_completed',
 'variant_count': 2,
 'analysis_manifest_preflight_status': 'needs_manifest',
 'analysis_manifest_preflight_action': 'review_backfill',
 'analysi

## 自动回填模式：dashboard export 前补 manifest

显式打开 `analysis_manifest_auto_backfill=True` 后，schedule 在 dashboard export 之前生成 manifest，dashboard payload 应显示 `analysis_manifest` 来源。

In [4]:
auto_config = ModelDashboardPipelineConfig(
    name='schedule_manifest_auto_backfill',
    card_dir=AUTO_CARD,
    dashboard_output_path=CARD / 'auto_dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    analysis_manifest_preflight=True,
    analysis_manifest_auto_backfill=True,
    analysis_manifest_dataset_name='schedule_preflight_demo',
    analysis_manifest_universe=('SOXL', 'SOXX'),
    analysis_manifest_calendar_path='data/qlib/soxl/calendars/day.txt',
)
auto_result = run_model_dashboard_refresh_once(
    auto_config,
    lock_dir=LOCKS,
    matrix_runner=legacy_matrix_runner,
)
auto_manifest = json.loads((AUTO_CARD / 'analysis_manifest.json').read_text(encoding='utf-8'))
auto_dashboard = json.loads((CARD / 'auto_dashboard_export.json').read_text(encoding='utf-8'))
assert auto_result.status == 'success'
assert auto_result.payload['analysis_manifest_preflight_status'] == 'backfilled'
assert auto_dashboard['manifest_source']['kind'] == 'analysis_manifest'
assert auto_dashboard['manifest_source']['manifest_hash'] == auto_manifest['manifest_hash']
assert 'predictions' not in auto_manifest['paths']
auto_result.payload

{'matrix_card': '/Users/rziiiii/dev/workspace/research/schedule_analysis_manifest_preflight_v1/auto_card',
 'dashboard_export': '/Users/rziiiii/dev/workspace/research/schedule_analysis_manifest_preflight_v1/auto_dashboard_export.json',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_artifact_id': 'auto_card',
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_manifest_path': '/Users/rziiiii/dev/workspace/research/schedule_analysis_manifest_preflight_v1/auto_card/analysis_manifest.json',
 'dashboard_manifest_hash': 'b5eba5941f2fdc29214f11e343081eefaa05873f30122b7b936bff7d35ad1989',
 'dashboard_manifest_producer': 'src.analysis.manifest_backfill',
 'dashboard_page': '',
 'dashboard_page_schema_version': '',
 'dashboard_view_model_schema_version': '',
 'dashboard_page_model_card_count': 0,
 'top_model': 'ridge_mse',
 'attribution_card': '',
 'attribution_status': 'not_provided',
 'matrix_status': 'feature_set_matrix_completed',
 'variant_count': 2,
 'analys

In [5]:
summary = {
    'schema_version': 'research.schedule_analysis_manifest_preflight_v1',
    'status': 'completed',
    'audit_preflight_status': audit_result.payload['analysis_manifest_preflight_status'],
    'audit_dashboard_manifest_kind': audit_result.payload['dashboard_manifest_kind'],
    'auto_preflight_status': auto_result.payload['analysis_manifest_preflight_status'],
    'auto_dashboard_manifest_kind': auto_result.payload['dashboard_manifest_kind'],
    'auto_manifest_hash': auto_manifest['manifest_hash'],
    'private_sidecars_hidden': 'predictions' not in auto_manifest['paths'],
    'audit_dashboard_export': str((CARD / 'audit_dashboard_export.json').relative_to(REPO_ROOT)),
    'auto_dashboard_export': str((CARD / 'auto_dashboard_export.json').relative_to(REPO_ROOT)),
    'conclusion': 'schedule dashboard refresh 已能在导出前审计 analysis_manifest，并可显式回填旧 card。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'preflight_results.json').write_text(json.dumps({'audit': audit_result.payload, 'auto': auto_result.payload}, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.schedule_analysis_manifest_preflight_v1',
 'status': 'completed',
 'audit_preflight_status': 'needs_manifest',
 'audit_dashboard_manifest_kind': 'fixed_sidecars',
 'auto_preflight_status': 'backfilled',
 'auto_dashboard_manifest_kind': 'analysis_manifest',
 'auto_manifest_hash': 'b5eba5941f2fdc29214f11e343081eefaa05873f30122b7b936bff7d35ad1989',
 'private_sidecars_hidden': True,
 'audit_dashboard_export': 'research/schedule_analysis_manifest_preflight_v1/audit_dashboard_export.json',
 'auto_dashboard_export': 'research/schedule_analysis_manifest_preflight_v1/auto_dashboard_export.json',
 'conclusion': 'schedule dashboard refresh 已能在导出前审计 analysis_manifest，并可显式回填旧 card。'}

## 中文结论

- 做了什么：把 analysis manifest preflight 接进 schedule 的 dashboard refresh 链路，并验证审计模式与显式自动回填模式。
- 核心指标：审计模式返回 `needs_manifest` 且不写 `analysis_manifest.json`；自动模式返回 `backfilled`，dashboard manifest 来源变为 `analysis_manifest`。
- regime 差异：旧 card 的 `regime_metrics.csv` 在自动回填后进入公开 manifest，dashboard 可继续展示 bull/stress 切片；`predictions.csv` 不进入 manifest。
- 是否晋升：这是 schedule 框架能力，可保留在 `src/schedule`；不是策略晋升。
- 风险：默认只审计，不自动覆盖历史产物；只有显式设置 auto backfill 时才写 manifest。
- 下一步：可把 preflight payload 接入通知文案，提醒哪些 card 仍然是 fixed-sidecar fallback。